# WholeBrain Schaefer MVPA Output Visualization
Ordered report for saved outputs from the `mvpa_L2_*` scripts. The notebook loads checkpoint/intermediate files from `/Users/xiaoqianxiao/projects/NARSAD/MRI/derivatives/fMRI_analysis/LSS/results/wholebrain_parcellation_schaefer` and does not rerun heavy modeling stages.


## Setup


In [ ]:
import os
import math
import glob
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Image, Markdown
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

PIPELINE_NAME = "wholebrain_parcellation_schaefer"
PIPELINE_LABEL = "WholeBrain Schaefer"
OUT_DIR = Path(os.path.join("/Users/xiaoqianxiao/projects/NARSAD/MRI/derivatives/fMRI_analysis/LSS/results", PIPELINE_NAME))
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
LEGACY_CHECKPOINT_DIR = OUT_DIR / "checkpoint"
INTERMEDIATE_DIR = OUT_DIR / "intermediate"
FIGURE_DIR = OUT_DIR / "notebook_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Pipeline:", PIPELINE_LABEL)
print("OUT_DIR:", OUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("LEGACY_CHECKPOINT_DIR:", LEGACY_CHECKPOINT_DIR)
print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR)
print("Notebook figures will be saved to:", FIGURE_DIR)


## Helper Functions


In [ ]:
def _existing(paths):
    for path in paths:
        p = Path(path)
        if p.exists():
            return p
    return None


def load_joblib(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        obj = joblib.load(path)
        plt.close("all")
        return obj
    except Exception as exc:
        print(f"Could not load {path}: {exc}")
        plt.close("all")
        return default


def load_first(paths, default=None):
    for path in paths:
        obj = load_joblib(path, default=None)
        if obj is not None:
            print(f"Loaded: {path}")
            return obj
    return default


def coerce_result(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        for key in ["results", "result", "payload", "data"]:
            if key in obj and isinstance(obj[key], dict):
                merged = dict(obj[key])
                for k, v in obj.items():
                    if k not in merged:
                        merged[k] = v
                return merged
        return obj
    return {"value": obj}


def first_present(mapping, keys):
    if mapping is None:
        return None
    for key in keys:
        if isinstance(mapping, dict) and key in mapping and mapping[key] is not None:
            return mapping[key]
    return None


def flatten_result_dict(d, prefix=""):
    rows = []
    if d is None:
        return pd.DataFrame(columns=["key", "value"])
    if not isinstance(d, dict):
        return pd.DataFrame([{"key": prefix or "value", "value": repr(d)}])
    for key, value in d.items():
        name = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, dict):
            rows.extend(flatten_result_dict(value, name).to_dict("records"))
        elif isinstance(value, (str, int, float, bool, np.integer, np.floating)) or value is None:
            rows.append({"key": name, "value": value})
        elif isinstance(value, pd.DataFrame):
            rows.append({"key": name, "value": f"DataFrame shape={value.shape}"})
        elif isinstance(value, np.ndarray):
            rows.append({"key": name, "value": f"ndarray shape={value.shape}, dtype={value.dtype}"})
        elif isinstance(value, (list, tuple)):
            rows.append({"key": name, "value": f"{type(value).__name__} len={len(value)}"})
        else:
            rows.append({"key": name, "value": type(value).__name__})
    return pd.DataFrame(rows)


def display_df(df, rows=30):
    if df is None:
        display(Markdown("_No table available._"))
        return
    if isinstance(df, dict):
        df = flatten_result_dict(df)
    if not isinstance(df, pd.DataFrame):
        display(pd.DataFrame({"value": [repr(df)]}))
        return
    if df.empty:
        display(Markdown("_Table is empty._"))
        return
    display(df.head(rows))


def find_figures(*keywords):
    roots = [OUT_DIR, CHECKPOINT_DIR, LEGACY_CHECKPOINT_DIR, INTERMEDIATE_DIR]
    figs = []
    key_l = [k.lower() for k in keywords if k]
    for root in roots:
        if not root.exists():
            continue
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.svg"):
            for path in root.rglob(ext):
                name = path.name.lower()
                if not key_l or any(k in name for k in key_l):
                    figs.append(path)
    return sorted(set(figs))


def show_figures_by_keywords(title, *keywords, max_figs=12):
    figs = find_figures(*keywords)
    display(Markdown(f"**Saved figures: {title}**"))
    if not figs:
        display(Markdown("_No saved figures matched these keywords._"))
        return []
    for path in figs[:max_figs]:
        display(Markdown(f"`{path}`"))
        if path.suffix.lower() == ".svg":
            display(Markdown(f"![{path.name}]({path})"))
        else:
            display(Image(filename=str(path)))
    if len(figs) > max_figs:
        display(Markdown(f"_Showing {max_figs} of {len(figs)} matched figures._"))
    return figs


def save_current_fig(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved {path}")


def plot_matrix(mat, title, labels=None, ax=None, cmap="vlag", center=0):
    arr = np.asarray(mat)
    if arr.ndim == 3:
        arr = np.nanmean(arr, axis=0)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D/3D matrix, got shape {arr.shape}")
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(arr, ax=ax, cmap=cmap, center=center, square=True, annot=True, fmt=".3g", xticklabels=labels, yticklabels=labels, cbar_kws={"shrink": 0.75})
    ax.set_title(title)
    return ax


def maybe_plot_numeric_bars(df, title, max_cols=12):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return False
    numeric = df.select_dtypes(include=[np.number])
    if numeric.empty:
        return False
    means = numeric.mean(numeric_only=True).sort_values(key=lambda s: s.abs(), ascending=False).head(max_cols)
    fig, ax = plt.subplots(figsize=(max(7, len(means) * 0.65), 4))
    sns.barplot(x=means.index, y=means.values, ax=ax, color="#4C78A8")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Mean")
    save_current_fig(title.lower().replace(" ", "_").replace("/", "_") + ".png")
    plt.show()
    return True


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full(pvals.shape, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    p = pvals[valid]
    if p.size == 0:
        return out
    order = np.argsort(p)
    ranked = p[order] * p.size / np.arange(1, p.size + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    ranked = np.clip(ranked, 0, 1)
    q = np.empty_like(ranked)
    q[order] = ranked
    out[valid] = q
    return out


def welch_by_trial(df, value_col, group_col="Group", trial_col="trial"):
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame()
    needed = {value_col, group_col, trial_col}
    if not needed.issubset(df.columns):
        return pd.DataFrame()
    rows = []
    for trial, sub in df.groupby(trial_col):
        sad = pd.to_numeric(sub.loc[sub[group_col].astype(str).str.upper().eq("SAD"), value_col], errors="coerce").dropna()
        hc = pd.to_numeric(sub.loc[sub[group_col].astype(str).str.upper().eq("HC"), value_col], errors="coerce").dropna()
        if len(sad) >= 2 and len(hc) >= 2:
            t, p = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit")
            pooled = math.sqrt(((sad.std(ddof=1) ** 2) + (hc.std(ddof=1) ** 2)) / 2) if len(sad) > 1 and len(hc) > 1 else np.nan
            d = (sad.mean() - hc.mean()) / pooled if pooled and np.isfinite(pooled) and pooled > 0 else np.nan
        else:
            t, p, d = np.nan, np.nan, np.nan
        rows.append({"trial": trial, "n_sad": len(sad), "n_hc": len(hc), "mean_sad": sad.mean() if len(sad) else np.nan, "mean_hc": hc.mean() if len(hc) else np.nan, "t": t, "p": p, "cohens_d": d})
    out = pd.DataFrame(rows).sort_values("trial") if rows else pd.DataFrame()
    if not out.empty:
        out["p_fdr_bh"] = bh_fdr(out["p"].to_numpy())
    return out


def extract_first_dataframe(result, preferred=()):
    if not isinstance(result, dict):
        return None
    for key in preferred:
        val = result.get(key)
        if isinstance(val, pd.DataFrame):
            return val
    for val in result.values():
        if isinstance(val, pd.DataFrame):
            return val
    return None


def plot_corr_table(df, title):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        display(Markdown("_No correlation table available._"))
        return
    display_df(df, rows=40)
    possible_r = [c for c in df.columns if c.lower() in {"r", "rho", "corr", "correlation", "estimate"}]
    x_cols = [c for c in df.columns if any(s in c.lower() for s in ["neural", "metric", "index", "predictor"])]
    y_cols = [c for c in df.columns if any(s in c.lower() for s in ["clinical", "score", "outcome", "measure"])]
    if possible_r and x_cols and y_cols:
        try:
            piv = df.pivot_table(index=y_cols[0], columns=x_cols[0], values=possible_r[0], aggfunc="mean")
            fig, ax = plt.subplots(figsize=(max(7, 0.45 * piv.shape[1]), max(4, 0.35 * piv.shape[0])))
            sns.heatmap(piv, cmap="vlag", center=0, ax=ax, cbar_kws={"label": possible_r[0]})
            ax.set_title(title)
            save_current_fig(title.lower().replace(" ", "_").replace("/", "_") + ".png")
            plt.show()
        except Exception as exc:
            print(f"Could not make correlation heatmap: {exc}")


## Output Manifest


In [ ]:
# Load saved stage/checkpoint payloads. Missing files are expected if a stage has not run yet.
paths = {
    "stage03_data": [CHECKPOINT_DIR / "cell_03.joblib", LEGACY_CHECKPOINT_DIR / "cell_03.joblib"],
    "stage06_results_11": [CHECKPOINT_DIR / "results_analysis_11.joblib", CHECKPOINT_DIR / "results_11.joblib", CHECKPOINT_DIR / "cell_06.joblib", LEGACY_CHECKPOINT_DIR / "results_analysis_11.joblib", LEGACY_CHECKPOINT_DIR / "results_11.joblib", LEGACY_CHECKPOINT_DIR / "cell_06.joblib"],
    "stage10_haufe": [INTERMEDIATE_DIR / "stage10_haufe_maps.joblib", CHECKPOINT_DIR / "cell_10.joblib", LEGACY_CHECKPOINT_DIR / "cell_10.joblib"],
    "stage11_importance": [INTERMEDIATE_DIR / "stage11_importance_masks.joblib", CHECKPOINT_DIR / "cell_11.joblib"],
    "stage12_topology": [CHECKPOINT_DIR / "analysis_12_topology.joblib", CHECKPOINT_DIR / "results_analysis_12.joblib", INTERMEDIATE_DIR / "stage12_topology_stats.joblib", INTERMEDIATE_DIR / "stage12_StaticRepresentationalTopology.joblib", INTERMEDIATE_DIR / "stage12_topology.joblib", CHECKPOINT_DIR / "results_12.joblib", CHECKPOINT_DIR / "cell_12.joblib", CHECKPOINT_DIR / "cell_10.joblib"],
    "stage13_drift": [CHECKPOINT_DIR / "results_analysis_13.joblib", INTERMEDIATE_DIR / "stage13_drift.joblib", CHECKPOINT_DIR / "cell_13.joblib"],
    "stage14_trajectories": [CHECKPOINT_DIR / "cell_12_trajectories.joblib", CHECKPOINT_DIR / "results_analysis_13_trajectories.joblib", INTERMEDIATE_DIR / "stage14_trajectories.joblib", CHECKPOINT_DIR / "cell_14.joblib"],
    "stage15_decision": [CHECKPOINT_DIR / "results_analysis_14.joblib", INTERMEDIATE_DIR / "stage15_decision_boundary.joblib", CHECKPOINT_DIR / "cell_15.joblib"],
    "stage16_safety_threat": [CHECKPOINT_DIR / "results_analysis_21.joblib", INTERMEDIATE_DIR / "stage16_safety_threat.joblib", CHECKPOINT_DIR / "cell_16.joblib"],
    "stage18_drift_efficiency": [CHECKPOINT_DIR / "results_analysis_22.joblib", INTERMEDIATE_DIR / "stage18_drift_efficiency.joblib", CHECKPOINT_DIR / "cell_18.joblib"],
    "stage19_prob_opening": [CHECKPOINT_DIR / "results_analysis_23.joblib", INTERMEDIATE_DIR / "stage19_probabilistic_opening.joblib", CHECKPOINT_DIR / "cell_19.joblib"],
    "stage20_realignment": [CHECKPOINT_DIR / "results_analysis_24.joblib", INTERMEDIATE_DIR / "stage20_spatial_realignment.joblib", CHECKPOINT_DIR / "cell_20.joblib"],
    "stage21_reverse": [CHECKPOINT_DIR / "results_analysis_25.joblib", INTERMEDIATE_DIR / "stage21_reverse_cross_decoding.joblib", CHECKPOINT_DIR / "cell_21.joblib"],
    "stage23_clinical_scores": [INTERMEDIATE_DIR / "stage23_clinical_scores.joblib", CHECKPOINT_DIR / "cell_23.joblib"],
    "stage24_neural_indices": [INTERMEDIATE_DIR / "stage24_neural_clinical_indices.joblib", CHECKPOINT_DIR / "cell_24.joblib"],
    "stage26_master_merge": [INTERMEDIATE_DIR / "stage26_master_neural_clinical.joblib", CHECKPOINT_DIR / "cell_26.joblib"],
    "stage27_pearson": [INTERMEDIATE_DIR / "stage27_neural_clinical_pearson.joblib", CHECKPOINT_DIR / "cell_27.joblib"],
    "stage28_partial": [INTERMEDIATE_DIR / "stage28_neural_clinical_partial.joblib", CHECKPOINT_DIR / "cell_28.joblib"],
    "stage29_zscore_outlier": [INTERMEDIATE_DIR / "stage29_zscore_outlier.joblib", CHECKPOINT_DIR / "cell_29.joblib"],
    "stage30_ols": [INTERMEDIATE_DIR / "stage30_z_ols_regression.joblib", CHECKPOINT_DIR / "cell_30.joblib"],
}
loaded = {name: load_first(candidates, default=None) for name, candidates in paths.items()}
results = {name: coerce_result(payload) for name, payload in loaded.items()}
manifest = []
for name, candidates in paths.items():
    existing = _existing(candidates)
    manifest.append({"stage": name, "found": existing is not None, "path": str(existing) if existing else ""})
display_df(pd.DataFrame(manifest), rows=50)


## 1. Analysis 1.1: Neural Dissociation Self-Decoding
This section summarizes self-decoding accuracy with null distribution, functional specificity, and spatial specificity where those outputs were saved by the script.


In [ ]:
r11_payload = results.get("stage06_results_11", {})
r11 = r11_payload.get("results_11", r11_payload) if isinstance(r11_payload, dict) else {}
show_figures_by_keywords("Analysis 1.1 neural dissociation", "analysis_11", "results_11", "neural_dissociation", "functional_specificity", "spatial_specificity", max_figs=10)
display_df(flatten_result_dict(r11), rows=60)

# Recreate the four-panel Analysis 1.1 figure: self-decoding null distributions, functional specificity, and spatial specificity.
def _lookup_many(*keys):
    for key in keys:
        if isinstance(r11, dict) and key in r11 and r11[key] is not None:
            return r11[key]
        if isinstance(r11_payload, dict) and key in r11_payload and r11_payload[key] is not None:
            return r11_payload[key]
    return None


def _scalar_or_nan(value):
    try:
        arr = np.asarray(value, dtype=float)
        if arr.size == 1:
            return float(arr.ravel()[0])
    except Exception:
        pass
    return np.nan


def _format_p(p):
    p = _scalar_or_nan(p)
    if np.isfinite(p):
        return f"p = {p:.4f}"
    return "p = n/a"


def _plot_self_decoding(ax, null_dist, obs, p_val, title):
    obs = _scalar_or_nan(obs)
    null_arr = np.asarray(null_dist, dtype=float).ravel() if null_dist is not None else np.array([])
    null_arr = null_arr[np.isfinite(null_arr)]
    if null_arr.size:
        sns.histplot(null_arr, stat="density", bins=35, color="lightgray", edgecolor="black", alpha=0.85, ax=ax, label="Null Dist")
        try:
            sns.kdeplot(null_arr, ax=ax, color="gray", linewidth=2.5)
        except Exception:
            pass
        thresh = np.nanpercentile(null_arr, 95)
        ax.axvline(thresh, color="blue", linestyle="--", linewidth=2, label="95% null")
    else:
        ax.text(0.5, 0.5, "Null distribution not found", ha="center", va="center", transform=ax.transAxes)
    if np.isfinite(obs):
        ax.axvline(obs, color="red", linewidth=2.5, label=f"Obs: {obs:.2f}")
    ax.set_title(f"{title} (CV Acc: {obs:.2f})\n({_format_p(p_val)})" if np.isfinite(obs) else f"{title}\n({_format_p(p_val)})")
    ax.set_xlabel("Forced-choice accuracy")
    ax.set_ylabel("Density")
    ax.legend(frameon=True, fontsize=10)

acc_sad = _lookup_many("acc_sad_cv", "accuracy_sad", "sad_accuracy")
acc_hc = _lookup_many("acc_hc_cv", "accuracy_hc", "hc_accuracy")
p_sad = _lookup_many("p_sad", "p_acc_sad", "sad_p")
p_hc = _lookup_many("p_hc", "p_acc_hc", "hc_p")
perm_sad = _lookup_many("perm_dist_sad", "perm_acc_sad", "null_dist_sad", "sad_null")
perm_hc = _lookup_many("perm_dist_hc", "perm_acc_hc", "null_dist_hc", "hc_null")
func_matrix = _lookup_many("func_matrix", "functional_specificity", "functional_matrix")
func_pvals = _lookup_many("p_func_pvals", "func_pvals", "functional_pvals")
sim_spatial = _lookup_many("sim_spatial", "obs_sim", "patterm_similarity", "pattern_similarity")
p_sim = _lookup_many("p_sim", "p_sim_spatial", "p_spatial")
spatial_matrix = _lookup_many("spatial_matrix")
spatial_pvals = _lookup_many("spatial_pvals")

if func_matrix is None:
    func_matrix = np.array([[_scalar_or_nan(acc_sad), np.nan], [np.nan, _scalar_or_nan(acc_hc)]], dtype=float)
else:
    func_matrix = np.asarray(func_matrix, dtype=float)
if func_pvals is None:
    func_pvals = np.array([[_scalar_or_nan(p_sad), np.nan], [np.nan, _scalar_or_nan(p_hc)]], dtype=float)
else:
    func_pvals = np.asarray(func_pvals, dtype=float)
if spatial_matrix is None:
    sim = _scalar_or_nan(sim_spatial)
    spatial_matrix = np.array([[1.0, sim], [sim, 1.0]], dtype=float)
else:
    spatial_matrix = np.asarray(spatial_matrix, dtype=float)
if spatial_pvals is None:
    ps = _scalar_or_nan(p_sim)
    spatial_pvals = np.array([[0.0, ps], [ps, 0.0]], dtype=float)
else:
    spatial_pvals = np.asarray(spatial_pvals, dtype=float)

if any(x is not None for x in [perm_sad, perm_hc, func_matrix, spatial_matrix]):
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.25], hspace=0.38, wspace=0.28)
    _plot_self_decoding(fig.add_subplot(gs[0, 0]), perm_sad, acc_sad, p_sad, "SAD Self-Decoding")
    _plot_self_decoding(fig.add_subplot(gs[0, 1]), perm_hc, acc_hc, p_hc, "HC Self-Decoding")
    annot_func = np.empty(func_matrix.shape, dtype=object)
    for i in range(func_matrix.shape[0]):
        for j in range(func_matrix.shape[1]):
            val = func_matrix[i, j]
            pv = func_pvals[i, j] if func_pvals.shape == func_matrix.shape else np.nan
            star = "*" if np.isfinite(pv) and pv < 0.05 else ""
            annot_func[i, j] = f"{val:.3f}\n({star})" if np.isfinite(val) else "n/a"
    ax3 = fig.add_subplot(gs[1, 0])
    sns.heatmap(func_matrix, annot=annot_func, fmt="", cmap="RdBu_r", center=0.5, vmin=0.3, vmax=0.9, cbar=True, xticklabels=["Test SAD", "Test HC"], yticklabels=["Train SAD", "Train HC"], ax=ax3)
    ax3.set_title("Functional Specificity\n(Forced-Choice Accuracy)")
    annot_spatial = np.empty(spatial_matrix.shape, dtype=object)
    for i in range(spatial_matrix.shape[0]):
        for j in range(spatial_matrix.shape[1]):
            val = spatial_matrix[i, j]
            pv = spatial_pvals[i, j] if spatial_pvals.shape == spatial_matrix.shape else np.nan
            star = "*" if i != j and np.isfinite(pv) and pv < 0.05 else ""
            annot_spatial[i, j] = f"{val:.3f}\n{star}" if np.isfinite(val) else "n/a"
    ax4 = fig.add_subplot(gs[1, 1])
    sns.heatmap(spatial_matrix, annot=annot_spatial, fmt="", cmap="RdBu_r", center=0, vmin=-1, vmax=1, cbar=True, xticklabels=["SAD Map", "HC Map"], yticklabels=["SAD Map", "HC Map"], ax=ax4)
    ax4.set_title("Spatial Specificity")
    save_current_fig("section01_neural_dissociation_four_panel.png")
    plt.show()
else:
    display(Markdown("_Analysis 1.1 four-panel ingredients were not found in the saved payload._"))

for key in ["acc_summary", "accuracy_summary", "df_accuracy", "df_acc", "functional_specificity", "spatial_specificity", "null_distribution"]:
    val = r11.get(key) if isinstance(r11, dict) else None
    if isinstance(val, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(val, rows=30)
    elif isinstance(val, np.ndarray) and val.ndim in (1, 2):
        fig, ax = plt.subplots(figsize=(6, 4))
        if val.ndim == 1:
            ax.hist(val[np.isfinite(val)], bins=30, color="#4C78A8", alpha=0.8)
            ax.set_title(key)
        else:
            plot_matrix(val, key, ax=ax)
        save_current_fig(f"section01_{key}.png")
        plt.show()


## 2. Haufe / Z-Scored Maps
Glass-brain figures are shown if the scripts saved them. The table lists available Haufe or z-map payloads for traceability.


In [ ]:
r10 = results.get("stage10_haufe", {})
r03 = results.get("stage03_data", {})
r11_payload = results.get("stage06_results_11", {})
r11_for_maps = r11_payload.get("results_11", r11_payload) if isinstance(r11_payload, dict) else {}
display_df(flatten_result_dict(r10), rows=60)

# FearNetworkAll/MemoryFearNetwork-style glass-brain reconstruction for ROI permutation maps.
def _nested_get(mapping, keys):
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        if key in mapping and mapping[key] is not None:
            return mapping[key]
    for value in mapping.values():
        if isinstance(value, dict):
            found = _nested_get(value, keys)
            if found is not None:
                return found
    return None


def _pipeline_roi_candidates():
    if PIPELINE_NAME == "FearNetwork":
        roots = [
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/Gillian_anatomically_constrained"),
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/NARSAD_uni/Gillian_NARSAD"),
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/FearNetwork"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/FearNetwork"),
        ]
    elif PIPELINE_NAME == "MemoryFearNetwork":
        roots = [
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/MemoryFearNetwork"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/MemoryFearNetwork"),
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        ]
    else:
        roots = [
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI") / PIPELINE_NAME,
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs") / PIPELINE_NAME,
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        ]
    env_name = "MEMORY_FEAR_ROI_DIR" if PIPELINE_NAME == "MemoryFearNetwork" else "FEAR_ROI_DIR"
    env_path = os.environ.get(env_name) or os.environ.get("ROI_DIR")
    if env_path:
        roots.insert(0, Path(env_path))
    return [p for p in roots if p.exists()]


def _default_roi_order():
    rois = _nested_get(r03, ["TARGET_ROIS", "ROI_ORDER", "roi_order", "target_rois"])
    if rois is not None:
        return list(rois)
    if PIPELINE_NAME == "MemoryFearNetwork":
        return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "left_VVC", "left_AG", "left_SMG", "left_IFG", "left_MFG", "left_SFG", "left_Precuneus", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc", "right_VVC", "right_AG", "right_SMG", "right_IFG", "right_MFG", "right_SFG", "right_Precuneus"]
    return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc"]


def _find_roi_dir(roi_order):
    for roi_dir in _pipeline_roi_candidates():
        if all(list(roi_dir.glob(f"*{roi}*.nii*")) for roi in roi_order):
            return roi_dir
    return None


def _roi_chunk_lengths(roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception:
        return None, None, "nibabel unavailable"
    chunks = []
    ref_img = None
    for roi in roi_order:
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        if not files:
            return None, None, f"missing mask for {roi} in {roi_dir}"
        img = nib.load(str(files[0]))
        if ref_img is None:
            ref_img = img
        chunks.append(int(np.sum(img.get_fdata() > 0)))
    return chunks, ref_img, None


def _bh_mask(pvals, alpha=0.05):
    p = np.asarray(pvals, dtype=float).ravel()
    valid = np.isfinite(p)
    out = np.zeros(p.shape, dtype=bool)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    thresh = alpha * np.arange(1, len(ranked) + 1) / len(ranked)
    passed = ranked <= thresh
    if passed.any():
        cutoff = ranked[np.where(passed)[0].max()]
        out[valid] = pv <= cutoff
    return out


def _roi_fdr_mask(pvals, roi_order, roi_dir, alpha=0.05):
    chunks, _, err = _roi_chunk_lengths(roi_order, roi_dir)
    p = np.asarray(pvals, dtype=float).ravel()
    if chunks is None or sum(chunks) != p.size:
        return _bh_mask(p, alpha=alpha), "Global FDR fallback"
    mask = np.zeros(p.shape, dtype=bool)
    cursor = 0
    for n in chunks:
        mask[cursor: cursor + n] = _bh_mask(p[cursor: cursor + n], alpha=alpha)
        cursor += n
    return mask, "ROI-FDR"


def _load_perm_cache(group):
    candidates = [
        CHECKPOINT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        OUT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        CHECKPOINT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
        OUT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
    ]
    for path in candidates:
        obj = load_joblib(path, default=None)
        if isinstance(obj, dict) and {"obs_weights", "null_weights", "p_values_raw"}.issubset(obj.keys()):
            return obj, path
    return None, None


def _z_from_cache(cache):
    obs = np.asarray(cache["obs_weights"], dtype=float).ravel()
    null = np.asarray(cache["null_weights"], dtype=float)
    z = (obs - np.nanmean(null, axis=0)) / (np.nanstd(null, axis=0) + 1e-12)
    z = np.asarray(z, dtype=float).ravel()
    # The source notebooks flip CSR-first models so CSS/threat-positive evidence is red.
    if PIPELINE_NAME in {"FearNetwork", "MemoryFearNetwork"}:
        z = -z
    return z


def _matched_perm_maps():
    roi_order = _default_roi_order()
    roi_dir = _find_roi_dir(roi_order)
    caches = {}
    for group in ["SAD", "HC"]:
        cache, path = _load_perm_cache(group)
        if cache is not None:
            caches[group] = {"cache": cache, "path": path, "z": _z_from_cache(cache), "p": np.asarray(cache["p_values_raw"], dtype=float).ravel()}
    if set(caches) != {"SAD", "HC"}:
        return {}, [f"Permutation caches found for groups: {sorted(caches)}"]
    fdr_masks = {}
    fdr_counts = {}
    mode0 = {}
    for group, payload in caches.items():
        if roi_dir is not None:
            fdr_masks[group], mode0[group] = _roi_fdr_mask(payload["p"], roi_order, roi_dir)
        else:
            fdr_masks[group], mode0[group] = _bh_mask(payload["p"]), "Global FDR; ROI masks not found"
        fdr_counts[group] = int(np.sum(fdr_masks[group]))
    if fdr_counts["SAD"] > 0 and fdr_counts["HC"] > 0:
        match_cfg = {"SAD": None, "HC": None}
    elif fdr_counts["SAD"] > 0 and fdr_counts["HC"] == 0:
        match_cfg = {"SAD": None, "HC": fdr_counts["SAD"]}
    elif fdr_counts["HC"] > 0 and fdr_counts["SAD"] == 0:
        match_cfg = {"SAD": fdr_counts["HC"], "HC": None}
    else:
        match_cfg = {"SAD": "fallback", "HC": "fallback"}
    maps = {}
    messages = [f"ROI_DIR={roi_dir}", f"Initial ROI-FDR counts: SAD={fdr_counts['SAD']}, HC={fdr_counts['HC']}"]
    for group, payload in caches.items():
        z = payload["z"]
        cfg = match_cfg[group]
        if cfg is None:
            sig_mask = fdr_masks[group]
            mode = mode0[group]
        elif cfg == "fallback":
            cutoff = np.nanpercentile(np.abs(z[np.isfinite(z)]), 98)
            sig_mask = np.isfinite(z) & (np.abs(z) >= cutoff)
            mode = "Top 2% Fallback"
        else:
            top_idx = np.argsort(np.abs(z))[-int(cfg):]
            sig_mask = np.zeros(z.shape, dtype=bool)
            sig_mask[top_idx] = True
            mode = f"Top {int(cfg)} (Matched)"
        z_masked = z * sig_mask
        if PIPELINE_NAME == "FearNetwork":
            # Match the reference figure style: positive directional z-values on a red scale.
            z_masked = np.where(z_masked > 0, z_masked, 0.0)
        maps[group] = {"values": z_masked, "mode": mode, "path": payload["path"], "n": int(np.sum(sig_mask)), "roi_order": roi_order, "roi_dir": roi_dir}
    return maps, messages


def _as_img_like(obj):
    if obj is None:
        return None
    try:
        import nibabel as nib
        from nilearn import image
        if isinstance(obj, nib.spatialimages.SpatialImage):
            return obj
        if isinstance(obj, (str, Path)) and Path(obj).exists():
            return image.load_img(str(obj))
    except Exception:
        return None
    return None


def _reconstruct_roi_map(flat_data, roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception as exc:
        return None, f"nibabel unavailable: {exc}"
    chunks, ref_img, err = _roi_chunk_lengths(roi_order, roi_dir)
    if chunks is None:
        return None, err
    arr = np.asarray(flat_data, dtype=float).ravel()
    if sum(chunks) != arr.size:
        return None, f"ROI mask voxel count {sum(chunks)} does not match map length {arr.size}"
    final_vol = np.zeros(ref_img.shape, dtype=float)
    cursor = 0
    for roi, n in zip(roi_order, chunks):
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        mask_img = nib.load(str(files[0]))
        mask_data = mask_img.get_fdata() > 0
        final_vol[mask_data] = arr[cursor: cursor + n]
        cursor += n
    return nib.Nifti1Image(final_vol, ref_img.affine, ref_img.header), None


def _vector_to_img(values, map_info=None):
    try:
        import nibabel as nib
        from nilearn import masking
    except Exception as exc:
        return None, f"nilearn/nibabel unavailable: {exc}"
    img = _as_img_like(values)
    if img is not None:
        return img, None
    arr = np.asarray(values, dtype=float).ravel() if values is not None else np.array([])
    if arr.size == 0:
        return None, "map vector not found"
    if isinstance(map_info, dict) and map_info.get("roi_dir") is not None:
        img, err = _reconstruct_roi_map(arr, map_info["roi_order"], map_info["roi_dir"])
        if img is not None:
            return img, None
    mask_source = _nested_get(r10, ["mask_img", "brain_mask", "mask_ext", "roi_mask", "wholebrain_mask"])
    if mask_source is None:
        mask_source = _nested_get(r11_for_maps, ["mask_img", "brain_mask", "mask_ext", "roi_mask", "wholebrain_mask"])
    mask_img = _as_img_like(mask_source)
    if mask_img is not None:
        try:
            if int(mask_img.get_fdata().astype(bool).sum()) == arr.size:
                return masking.unmask(arr, mask_img), None
        except Exception as exc:
            return None, f"mask unmask failed: {exc}"
    return None, f"map has {arr.size} values but no compatible ROI masks or whole-brain mask were found"


def _stage_map(group):
    g = group.lower()
    candidates = [f"z_map_{g}", f"z_scores_{g}", f"z_haufe_{g}", f"haufe_z_{g}", f"map_{g}_z", f"map_{g}", f"haufe_{g}", f"pattern_{g}"]
    found = _nested_get(r10, candidates)
    if found is None:
        found = _nested_get(r11_for_maps, candidates)
    return {"values": found, "mode": "Haufe/Z map from stage output", "roi_order": _default_roi_order(), "roi_dir": _find_roi_dir(_default_roi_order())}


def _plot_glass(group, map_info, ax=None):
    try:
        from nilearn import plotting
    except Exception as exc:
        return False, f"nilearn plotting unavailable: {exc}"
    img, err = _vector_to_img(map_info.get("values"), map_info)
    if img is None:
        return False, err
    data = np.asarray(img.get_fdata(), dtype=float)
    finite = data[np.isfinite(data)]
    finite_nonzero = finite[finite != 0]
    if finite_nonzero.size == 0:
        return False, "map image contains no non-zero finite values"
    vmax = float(np.nanpercentile(np.abs(finite_nonzero), 99))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(np.abs(finite_nonzero)))
    threshold = float(np.nanmin(np.abs(finite_nonzero)))
    cmap = "Reds" if PIPELINE_NAME == "FearNetwork" else "RdBu_r"
    title = f"{group} Fear Network: {map_info.get('mode')}" if PIPELINE_NAME == "FearNetwork" else f"{group} {PIPELINE_LABEL}: {map_info.get('mode')}"
    plotting.plot_glass_brain(img, display_mode="lyrz", colorbar=True, threshold=threshold, vmax=vmax, cmap=cmap, plot_abs=False, black_bg=False, axes=ax, title=title)
    return True, map_info.get("mode")

matched_maps, match_messages = _matched_perm_maps()
if matched_maps:
    fig = plt.figure(figsize=(13, 5))
    axes = {"SAD": fig.add_subplot(1, 2, 1), "HC": fig.add_subplot(1, 2, 2)}
    messages = []
    rendered = []
    for group, ax in axes.items():
        ok, msg = _plot_glass(group, matched_maps[group], ax=ax)
        rendered.append(ok)
        if not ok:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{group} glass brain could not be reconstructed\n{msg}", ha="center", va="center", wrap=True)
            messages.append(f"{group}: {msg}")
    save_current_fig("section02_fearnetwork_roi_fdr_glass_brain.png")
    plt.show()
    if messages:
        display(Markdown("; ".join(messages)))
    display_df(pd.DataFrame([{ "group": g, "mode": v["mode"], "selected_voxels": v["n"], "cache": str(v["path"]), "roi_dir": str(v["roi_dir"]) } for g, v in matched_maps.items()]), rows=10)
    display(Markdown("; ".join(match_messages)))
else:
    display(Markdown("_Permutation-cache ROI glass brain was unavailable; falling back to stage Haufe/Z maps._"))
    display(Markdown("; ".join(match_messages)))
    fig = plt.figure(figsize=(13, 5))
    axes = {"SAD": fig.add_subplot(1, 2, 1), "HC": fig.add_subplot(1, 2, 2)}
    rendered = []
    messages = []
    for group, ax in axes.items():
        ok, msg = _plot_glass(group, _stage_map(group), ax=ax)
        rendered.append(ok)
        if not ok:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{group} glass brain could not be reconstructed\n{msg}", ha="center", va="center", wrap=True)
            messages.append(f"{group}: {msg}")
    if any(rendered):
        save_current_fig("section02_haufe_z_glass_brain.png")
        plt.show()
    else:
        plt.close(fig)
        display(Markdown("_Could not reconstruct the Haufe/Z glass-brain figure from saved arrays in this local notebook._"))
        display(Markdown("Missing/issue details: " + "; ".join(messages)))
        show_figures_by_keywords("Saved Haufe/glass-brain fallback", "haufe", "zscore", "z_score", "glass", "activation_pattern", max_figs=4)


## 3. Permutation-Significant Voxels
This section reports permutation-importance feature-mask diagnostics and any saved glass-brain/ribbon figures for significant or fallback feature spaces.


In [ ]:
r11imp = results.get("stage11_importance", {})
show_figures_by_keywords("Permutation significant voxels", "stage11", "importance", "permutation", "permut", "significant", "river", max_figs=12)
display_df(flatten_result_dict(r11imp), rows=80)
rows = []
if isinstance(r11imp, dict):
    for group in ["SAD", "HC", "sad", "hc"]:
        obj = r11imp.get(group)
        if isinstance(obj, dict):
            mask = first_present(obj, ["significant_mask", "sig_mask", "mask", "importance_mask", "primary_mask"])
            imp = first_present(obj, ["importance", "importance_mean", "observed_importance", "actual_importance", "scores"])
            p = first_present(obj, ["p_values", "pvals", "p_uncorrected", "p"])
            q = first_present(obj, ["q_values", "qvals", "p_fdr", "fdr_p"])
            rows.append({"group": group.upper(), "mask_features": int(np.asarray(mask).sum()) if mask is not None else np.nan, "positive_importance": int((np.asarray(imp) > 0).sum()) if imp is not None else np.nan, "max_importance": float(np.nanmax(imp)) if imp is not None and np.asarray(imp).size else np.nan, "min_p": float(np.nanmin(p)) if p is not None and np.asarray(p).size else np.nan, "min_q": float(np.nanmin(q)) if q is not None and np.asarray(q).size else np.nan})
        elif obj is not None:
            arr = np.asarray(obj)
            rows.append({"group": group.upper(), "mask_features": int(arr.sum()) if arr.dtype == bool else np.nan, "positive_importance": int((arr > 0).sum()) if arr.dtype != bool else np.nan, "max_importance": float(np.nanmax(arr)) if arr.size else np.nan, "min_p": np.nan, "min_q": np.nan})
summary = pd.DataFrame(rows).drop_duplicates("group") if rows else pd.DataFrame()
display_df(summary, rows=10)
if not summary.empty and "mask_features" in summary:
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(data=summary, x="group", y="mask_features", ax=ax, color="#59A14F")
    ax.set_title("Permutation feature counts")
    ax.set_ylabel("Selected voxels")
    save_current_fig("section03_permutation_feature_counts.png")
    plt.show()


## 4. Static Representational Topology
Statistics are listed first, followed by SAD and HC RDM heatmaps and distance summaries where the saved topology payload contains them.


In [ ]:
r12 = results.get("stage12_topology", {})
show_figures_by_keywords("Static topology", "analysis_12", "results_12", "topology", "rdm", max_figs=12)
display_df(flatten_result_dict(r12), rows=80)
labels = first_present(r12, ["rdm_conditions", "conditions", "RDM_CONDITIONS"])
if labels is None:
    labels = ["CS-", "CSS", "CSR"]
sad_rdm = first_present(r12, ["rdms_sad_raw_pv", "rdms_sad_pv", "rdms_sad_raw", "rdms_sad", "rdm_sad"])
hc_rdm = first_present(r12, ["rdms_hc_raw_pv", "rdms_hc_pv", "rdms_hc_raw", "rdms_hc", "rdm_hc"])
if sad_rdm is not None or hc_rdm is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    if sad_rdm is not None:
        plot_matrix(sad_rdm, "SAD mean RDM", labels=labels, ax=axes[0])
    else:
        axes[0].axis("off")
    if hc_rdm is not None:
        plot_matrix(hc_rdm, "HC mean RDM", labels=labels, ax=axes[1])
    else:
        axes[1].axis("off")
    save_current_fig("section04_static_topology_heatmaps.png")
    plt.show()
for key in ["distance_df", "df_distances", "df_topology", "rdm_distance_stats", "summary_df"]:
    df = r12.get(key) if isinstance(r12, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Topology distance summary: {key}")

def _rdm_value_from_r12(keys):
    for key in keys:
        value = r12.get(key) if isinstance(r12, dict) else None
        if value is not None:
            arr = np.asarray(value, dtype=float)
            if arr.ndim == 3 and arr.shape[1] >= 4 and arr.shape[2] >= 4:
                return arr, key
    return None, None

shock_labels = r12.get("RDM_CONDITIONS_SHOCK") or ["CS-", "CSS", "CSR", "Shock"]
rdms_sad_shock, sad_shock_key = _rdm_value_from_r12(["rdms_sad_shock_raw_pv", "rdms_sad_shock_pv", "rdms_sad_shock_raw", "rdms_sad_shock"])
rdms_hc_shock, hc_shock_key = _rdm_value_from_r12(["rdms_hc_shock_raw_pv", "rdms_hc_shock_pv", "rdms_hc_shock_raw", "rdms_hc_shock"])
if rdms_sad_shock is not None and rdms_hc_shock is not None:
    display(Markdown("**Shock-inclusive RDM**"))
    shock_i = list(shock_labels).index("Shock") if "Shock" in list(shock_labels) else len(shock_labels) - 1
    rows = []
    for cond_i, cond in enumerate(shock_labels):
        if cond_i == shock_i:
            continue
        sad_vec = rdms_sad_shock[:, shock_i, cond_i]
        hc_vec = rdms_hc_shock[:, shock_i, cond_i]
        t_val, p_val = stats.ttest_ind(sad_vec, hc_vec, equal_var=False, nan_policy="omit")
        rows.append({"Metric": f"Shock vs {cond}", "SAD_Mean": np.nanmean(sad_vec), "HC_Mean": np.nanmean(hc_vec), "Group_t": t_val, "Group_p": p_val})
    display_df(pd.DataFrame(rows), rows=20)
    fig = plt.figure(figsize=(24, 8))
    gs = fig.add_gridspec(1, 3)
    ax1 = fig.add_subplot(gs[0, 0])
    sns.heatmap(np.nanmean(rdms_sad_shock, axis=0), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1.2, xticklabels=shock_labels, yticklabels=shock_labels, ax=ax1, cbar=False)
    ax1.set_title(f"SAD Shock-Inclusive Topology\n({sad_shock_key}; n={len(rdms_sad_shock)})")
    ax2 = fig.add_subplot(gs[0, 1])
    sns.heatmap(np.nanmean(rdms_hc_shock, axis=0), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1.2, xticklabels=shock_labels, yticklabels=shock_labels, ax=ax2)
    ax2.set_title(f"HC Shock-Inclusive Topology\n({hc_shock_key}; n={len(rdms_hc_shock)})")
    shock_df = []
    for cond_i, cond in enumerate(shock_labels):
        if cond_i == shock_i:
            continue
        shock_df.extend([{"Group": "SAD", "Metric": f"Shock-{cond}", "Distance": v} for v in rdms_sad_shock[:, shock_i, cond_i]])
        shock_df.extend([{"Group": "HC", "Metric": f"Shock-{cond}", "Distance": v} for v in rdms_hc_shock[:, shock_i, cond_i]])
    ax3 = fig.add_subplot(gs[0, 2])
    sns.violinplot(data=pd.DataFrame(shock_df), x="Metric", y="Distance", hue="Group", split=True, inner="quartile", palette={"SAD": "#c44e52", "HC": "#4c72b0"}, ax=ax3)
    ax3.set_title("Shock Distances")
    ax3.tick_params(axis="x", rotation=20)
    ax3.set_ylabel("Crossnobis Distance")
    plt.tight_layout()
    save_current_fig("schaefer_shock_inclusive_rdm.png")
    plt.show()
else:
    display(Markdown("_No shock-inclusive RDM found yet. Re-run Analysis 1.2/stage 12 after the shock-RDM update._"))



## 5. Dynamic Representational Drift
This section displays drift statistics and the drift figures saved by the script, then regenerates compact numeric summaries when possible.


In [ ]:
r13 = results.get("stage13_drift", {})
show_figures_by_keywords("Dynamic drift", "analysis_13_drift", "results_13_drift", "drift_plots", "drift", max_figs=12)
display_df(flatten_result_dict(r13), rows=80)
for key in ["df_plot", "df_drift", "drift_df", "stats_df", "summary_df"]:
    df = r13.get(key) if isinstance(r13, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Dynamic drift: {key}")


## 6. Single-Trial Safety and Threat Trajectories
Safety and threat are plotted separately because they may use different trajectory scores. Trial-wise SAD versus HC Welch tests are reported when trial-level data are available.


In [ ]:

r14 = results.get("stage14_trajectories", {})
if not isinstance(r14, dict) or not r14:
    r14 = results.get("stage13_DynamicRepresentationalDrift", {})
res132 = r14.get("results_13_2", r14) if isinstance(r14, dict) else {}
__ALLOW_DIRECT_TRAJECTORY_CACHE__ = True
if __ALLOW_DIRECT_TRAJECTORY_CACHE__ and not (isinstance(res132, dict) and "data_threat_shock" in res132):
    for _traj_path in [CHECKPOINT_DIR / "cell_12_trajectories.joblib", CHECKPOINT_DIR / "cell_14.joblib", INTERMEDIATE_DIR / "stage14_trajectories.joblib"]:
        if _traj_path.exists():
            try:
                _traj_payload = joblib.load(_traj_path)
            except Exception:
                continue
            _traj_res = _traj_payload.get("results_13_2", _traj_payload) if isinstance(_traj_payload, dict) else {}
            if isinstance(_traj_res, dict) and "data_threat_shock" in _traj_res:
                display(Markdown(f"_Loaded trajectory results directly from `{_traj_path}`._"))
                res132 = _traj_res
                break
elif not __ALLOW_DIRECT_TRAJECTORY_CACHE__ and not (isinstance(res132, dict) and res132.get("active_recomputed")):
    display(Markdown("_Variant notebook: section 6 uses notebook-side active-mask recomputation. Run section 3 first; cached trajectory joblibs are intentionally not used here._"))

stats_safe = res132.get("stats_safe") if isinstance(res132, dict) else None
stats_threat = res132.get("stats_threat") if isinstance(res132, dict) else None
stats_threat_shock = res132.get("stats_threat_shock") if isinstance(res132, dict) else None
df_safe = res132.get("data_safe") if isinstance(res132, dict) else None
df_threat = res132.get("data_threat") if isinstance(res132, dict) else None
df_threat_shock = res132.get("data_threat_shock") if isinstance(res132, dict) else None

display(Markdown("**Safety trajectory statistics**")); display_df(stats_safe, rows=30)
display(Markdown("**Threat maintenance trajectory statistics**")); display_df(stats_threat, rows=30)
display(Markdown("**Threat shock-target trajectory statistics**")); display_df(stats_threat_shock, rows=30)

def _format_p_for_plot(p):
    if pd.isna(p):
        return "NA"
    p = float(p)
    if p < 0.001:
        return "<.001"
    return f"={p:.3f}".replace("0.", ".")

def _annotate_sig_group_diffs(ax, data_df, stats_df):
    if not isinstance(data_df, pd.DataFrame) or data_df.empty or not isinstance(stats_df, pd.DataFrame) or stats_df.empty:
        return
    if "Trial" not in stats_df.columns:
        return
    use_col = None
    label_prefix = "p"
    if "Diff_p_fdr" in stats_df.columns and (pd.to_numeric(stats_df["Diff_p_fdr"], errors="coerce") < 0.05).any():
        use_col = "Diff_p_fdr"
        label_prefix = "q"
    elif "Diff_p" in stats_df.columns:
        use_col = "Diff_p"
    if use_col is None:
        return
    sig = stats_df.loc[pd.to_numeric(stats_df[use_col], errors="coerce") < 0.05].copy()
    if sig.empty:
        return
    base_y = 1.72
    step = 0.09
    for i, (_, row) in enumerate(sig.iterrows()):
        trial = float(row["Trial"])
        pval = float(row[use_col])
        y = min(base_y + (i % 3) * step, 1.95)
        ax.plot([trial - 0.22, trial + 0.22], [y - 0.045, y - 0.045], color="black", lw=1.6, clip_on=True)
        ax.text(trial, y, f"{label_prefix}{_format_p_for_plot(pval)}", ha="center", va="bottom", fontsize=14, color="black", clip_on=True)

trajectory_panels = [
    ("A. Safety Trajectory\n(Target = CS-)", df_safe, stats_safe, "o", "#2ca02c", "Target (CS-)", "Start (Fear)"),
    ("B. Threat Maintenance\n(Target = Reinstated CSR)", df_threat, stats_threat, "s", "#d62728", "Target (Reinstated CSR)", "Start (Ext Early)"),
    ("C. Threat Acquisition\n(Target = Shock/US)", df_threat_shock, stats_threat_shock, "^", "#9467bd", "Target (Shock/US)", "Start (Ext Early)"),
]
available_panels = [(title, df, stats_df, marker, target_color, target_label, start_label) for title, df, stats_df, marker, target_color, target_label, start_label in trajectory_panels if isinstance(df, pd.DataFrame) and not df.empty]
if available_panels:
    sns.set_context("poster")
    fig, axes = plt.subplots(1, len(available_panels), figsize=(11 * len(available_panels), 9), sharey=True, squeeze=False)
    axes = axes.ravel()
    for ax, (title, df_plot_panel, stats_panel, marker, target_color, target_label, start_label) in zip(axes, available_panels):
        sns.lineplot(data=df_plot_panel, x="trial", y="score", hue="Group", palette={"SAD": "#c44e52", "HC": "#4c72b0"}, lw=3, marker=marker, err_style="band", ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Trial (Block Size: 1)")
        ax.set_ylabel("Similarity Score (0=Start, 1=Target)")
        ax.axhline(0, color="gray", ls="--", label=start_label)
        ax.axhline(1, color=target_color, ls="-", lw=2, label=target_label)
        _annotate_sig_group_diffs(ax, df_plot_panel, stats_panel)
        ax.set_ylim(-1, 2)
        ax.legend(loc="upper left")
    save_current_fig(f"{PIPELINE_NAME.lower()}_stage14_trajectories_three_panel.png")
    plt.show()
else:
    display(Markdown("_No trajectory dataframes found. For variant notebooks, run section 3 first so trajectories are recomputed from ACTIVE_IMPORTANCE_MASKS._"))


## 6b. Trial-Wise SCR-Neural Coupling
This section correlates trial-wise SCR with single-trial neural trajectory scores when both saved outputs are available. Safety uses CSS SCR with safety trajectory score; threat uses CSR SCR with threat trajectory score.


In [ ]:
r14 = results.get("stage14_trajectories", {})
res132 = r14.get("results_13_2", r14) if isinstance(r14, dict) else {}
df_safe = res132.get('data_safe') if isinstance(res132, dict) else None
df_threat = res132.get('data_threat') if isinstance(res132, dict) else None
df_threat_shock = res132.get('data_threat_shock') if isinstance(res132, dict) else None
r23 = results.get("stage23_clinical_scores", {})
r24 = results.get("stage24_neural_indices", {})
r26 = results.get("stage26_master_merge", {})

def _first_df_from_payload(payload, keys):
    if not isinstance(payload, dict):
        return None
    for key in keys:
        val = payload.get(key)
        if isinstance(val, pd.DataFrame):
            return val
    return None

def _first_nonempty_df(*dfs):
    for df in dfs:
        if isinstance(df, pd.DataFrame):
            return df
    return None


df_scr_trials = _first_nonempty_df(
    _first_df_from_payload(r24, ["df_scr_trials"]),
    _first_df_from_payload(r23, ["df_scr_trials"]),
)
meta_for_scr = _first_nonempty_df(
    _first_df_from_payload(r23, ["meta"]),
    _first_df_from_payload(r24, ["meta"]),
    _first_df_from_payload(r26, ["meta"]),
)

def _normalize_sub_id(s):
    return str(s).replace("sub-", "").strip()

def _attach_meta(df):
    if not isinstance(df, pd.DataFrame) or df.empty or not isinstance(meta_for_scr, pd.DataFrame):
        return df
    meta = meta_for_scr.copy()
    sub_col = "subject_id" if "subject_id" in meta.columns else "sub_ID" if "sub_ID" in meta.columns else None
    if sub_col is None:
        return df
    keep = [sub_col] + [c for c in ["Group", "Drug", "drug_condition", "Gender", "demo_age"] if c in meta.columns]
    meta = meta[keep].drop_duplicates(sub_col).copy()
    meta["sub_ID"] = meta[sub_col].map(_normalize_sub_id)
    out = df.copy()
    out["sub_ID"] = out["sub_ID"].map(_normalize_sub_id)
    merge_cols = ["sub_ID"] + [c for c in meta.columns if c not in {sub_col, "sub_ID"}]
    out = out.merge(meta[merge_cols], on="sub_ID", how="left", suffixes=("", "_meta"))
    if "Group_meta" in out.columns and "Group" in out.columns:
        out["Group"] = out["Group"].fillna(out["Group_meta"])
        out = out.drop(columns=["Group_meta"])
    return out

def _compute_trialwise_scr_neural(neural_df, scr_df, scr_condition, domain):
    if not isinstance(neural_df, pd.DataFrame) or neural_df.empty or not isinstance(scr_df, pd.DataFrame) or scr_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    required_neural = {"sub", "trial", "score"}
    required_scr = {"sub_ID", "SCR_Condition", "condition_trial", "SCR_Anticipatory"}
    if not required_neural.issubset(neural_df.columns) or not required_scr.issubset(scr_df.columns):
        return pd.DataFrame(), pd.DataFrame()
    neural = neural_df.rename(columns={"sub": "sub_ID", "trial": "condition_trial", "score": "Neural_Trajectory_Score"}).copy()
    neural["sub_ID"] = neural["sub_ID"].map(_normalize_sub_id)
    neural["condition_trial"] = pd.to_numeric(neural["condition_trial"], errors="coerce")
    keep_cols = ["sub_ID", "condition_trial", "Neural_Trajectory_Score"] + [c for c in ["Group", "Condition"] if c in neural.columns]
    neural = neural[keep_cols]
    scr = scr_df[scr_df["SCR_Condition"].astype(str).eq(scr_condition)].copy()
    scr["sub_ID"] = scr["sub_ID"].map(_normalize_sub_id)
    scr["condition_trial"] = pd.to_numeric(scr["condition_trial"], errors="coerce")
    scr["SCR_Anticipatory"] = pd.to_numeric(scr["SCR_Anticipatory"], errors="coerce")
    merged = neural.merge(scr[["sub_ID", "condition_trial", "SCR_Anticipatory", "SCR_Trial"]], on=["sub_ID", "condition_trial"], how="inner")
    merged["Domain"] = domain
    merged["SCR_Condition"] = scr_condition
    merged = _attach_meta(merged)
    rows = []
    for sub_id, sub_df in merged.groupby("sub_ID"):
        valid = sub_df[["Neural_Trajectory_Score", "SCR_Anticipatory"]].dropna()
        r_val, p_val = np.nan, np.nan
        if len(valid) >= 3 and valid["Neural_Trajectory_Score"].nunique() > 1 and valid["SCR_Anticipatory"].nunique() > 1:
            r_val, p_val = stats.pearsonr(valid["Neural_Trajectory_Score"], valid["SCR_Anticipatory"])
        group = sub_df["Group"].dropna().iloc[0] if "Group" in sub_df.columns and sub_df["Group"].notna().any() else np.nan
        drug = sub_df["Drug"].dropna().iloc[0] if "Drug" in sub_df.columns and sub_df["Drug"].notna().any() else np.nan
        rows.append({"sub_ID": sub_id, "Group": group, "Drug": drug, "Domain": domain, "SCR_Condition": scr_condition, "r_neural_scr": r_val, "p_neural_scr": p_val, "n_trials": int(len(valid))})
    return merged, pd.DataFrame(rows)

merged_frames = []
coupling_frames = []
for domain, neural_df, scr_condition in [("Safety", df_safe, "CSS"), ("Threat", df_threat, "CSR"), ("Threat Shock Target", df_threat_shock, "CSR")]:
    merged, coupling = _compute_trialwise_scr_neural(neural_df, df_scr_trials, scr_condition, domain)
    if not merged.empty:
        merged_frames.append(merged)
    if not coupling.empty:
        coupling_frames.append(coupling)

if coupling_frames:
    trial_scr_neural = pd.concat(merged_frames, ignore_index=True) if merged_frames else pd.DataFrame()
    subject_scr_neural_coupling = pd.concat(coupling_frames, ignore_index=True)
    display(Markdown("**Subject-level trial-wise SCR-neural coupling**"))
    display_df(subject_scr_neural_coupling, rows=80)

    def _p_to_stars(p):
        try:
            p = float(p)
        except Exception:
            return ""
        if not np.isfinite(p):
            return ""
        if p < 0.001:
            return "***"
        if p < 0.01:
            return "**"
        if p < 0.05:
            return "*"
        if p < 0.10:
            return "+"
        return ""

    stats_rows = []
    for domain, sub_df in subject_scr_neural_coupling.groupby("Domain"):
        sad = pd.to_numeric(sub_df.loc[sub_df["Group"].astype(str).str.upper().eq("SAD"), "r_neural_scr"], errors="coerce").dropna()
        hc = pd.to_numeric(sub_df.loc[sub_df["Group"].astype(str).str.upper().eq("HC"), "r_neural_scr"], errors="coerce").dropna()
        if len(sad) >= 2 and len(hc) >= 2:
            t_val, p_val = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit")
        else:
            t_val, p_val = np.nan, np.nan
        stats_rows.append({"Domain": domain, "SAD_n": len(sad), "HC_n": len(hc), "SAD_mean_r": sad.mean() if len(sad) else np.nan, "HC_mean_r": hc.mean() if len(hc) else np.nan, "Welch_t": t_val, "p": p_val, "sig": _p_to_stars(p_val)})
    stats_table = pd.DataFrame(stats_rows)
    display(Markdown("**SAD vs HC tests of subject-level coupling**"))
    display_df(stats_table, rows=20)

    display(Markdown("**Primary trial-wise mixed-effects model**"))
    display(Markdown("Full-sample model: z(SCR) ~ z(neural trajectory score) * Group * Domain + Drug + trial + subject random effects. The code first tries a subject-level random neural-score slope and falls back to a random intercept if needed."))

    def _zscore_series(s):
        x = pd.to_numeric(s, errors="coerce")
        sd = x.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return x * np.nan
        return (x - x.mean()) / sd

    def _mixedlm_fixed_effects_table(fit, label, random_effects):
        conf = fit.conf_int()
        rows = []
        for term in fit.params.index:
            if term == "Group Var" or term.endswith(" Var") or " Cov" in term:
                continue
            rows.append({
                "model": label,
                "random_effects": random_effects,
                "term": term,
                "beta": fit.params.get(term, np.nan),
                "se": fit.bse.get(term, np.nan) if hasattr(fit, "bse") else np.nan,
                "z": fit.tvalues.get(term, np.nan) if hasattr(fit, "tvalues") else np.nan,
                "p": fit.pvalues.get(term, np.nan) if hasattr(fit, "pvalues") else np.nan,
                "sig": _p_to_stars(fit.pvalues.get(term, np.nan) if hasattr(fit, "pvalues") else np.nan),
                "ci_low": conf.loc[term, 0] if term in conf.index else np.nan,
                "ci_high": conf.loc[term, 1] if term in conf.index else np.nan,
                "n_trials": int(fit.nobs),
                "n_subjects": int(len(np.unique(fit.model.groups))),
            })
        return pd.DataFrame(rows)

    def _fit_trialwise_mixedlm(model_df, label, formula):
        try:
            import statsmodels.formula.api as smf
        except Exception as exc:
            return pd.DataFrame(), f"statsmodels unavailable for {label}: {exc}"
        dfm = model_df.copy()
        dfm = dfm.dropna(subset=["Neural_Trajectory_Score", "SCR_Anticipatory", "sub_ID", "Group", "Domain"])
        if dfm.empty or dfm["sub_ID"].nunique() < 3:
            return pd.DataFrame(), f"Not enough subjects for {label}."
        dfm["Neural_Z"] = _zscore_series(dfm["Neural_Trajectory_Score"])
        dfm["SCR_Z"] = _zscore_series(dfm["SCR_Anticipatory"])
        dfm["Trial_Z"] = _zscore_series(dfm["condition_trial"]) if "condition_trial" in dfm.columns else 0.0
        dfm["Group"] = dfm["Group"].astype(str).str.upper()
        dfm["Domain"] = dfm["Domain"].astype(str)
        if "Drug" not in dfm.columns:
            dfm["Drug"] = "Unknown"
        dfm["Drug"] = dfm["Drug"].fillna("Unknown").astype(str)
        dfm = dfm.dropna(subset=["Neural_Z", "SCR_Z", "Trial_Z"])
        if dfm.empty or dfm["SCR_Z"].nunique() < 2 or dfm["Neural_Z"].nunique() < 2:
            return pd.DataFrame(), f"Not enough variance for {label}."
        attempts = [("random intercept + neural-score slope", "~Neural_Z"), ("random intercept", "1")]
        last_error = None
        for random_label, re_formula in attempts:
            try:
                fit = smf.mixedlm(formula, data=dfm, groups=dfm["sub_ID"], re_formula=re_formula).fit(reml=False, method="lbfgs", maxiter=500, disp=False)
                return _mixedlm_fixed_effects_table(fit, label, random_label), ""
            except Exception as exc:
                last_error = exc
        return pd.DataFrame(), f"MixedLM failed for {label}: {last_error}"

    mixed_tables = []
    mixed_messages = []
    if not trial_scr_neural.empty:
        combined_formula = "SCR_Z ~ Neural_Z * C(Group) * C(Domain) + C(Drug) + Trial_Z"
        table, msg = _fit_trialwise_mixedlm(trial_scr_neural, "Full sample: SCR outcome", combined_formula)
        if not table.empty:
            mixed_tables.append(table)
        if msg:
            mixed_messages.append(msg)
        for domain, domain_df in trial_scr_neural.groupby("Domain"):
            domain_formula = "SCR_Z ~ Neural_Z * C(Group) + C(Drug) + Trial_Z"
            table, msg = _fit_trialwise_mixedlm(domain_df, f"{domain} only: SCR outcome", domain_formula)
            if not table.empty:
                mixed_tables.append(table)
            if msg:
                mixed_messages.append(msg)
    if mixed_tables:
        mixed_effects_table = pd.concat(mixed_tables, ignore_index=True)
        display_df(mixed_effects_table, rows=120)
        sig_mixed_effects = mixed_effects_table[pd.to_numeric(mixed_effects_table["p"], errors="coerce") < 0.05].copy()
        display(Markdown("**Significant mixed-effects terms (p < .05)**"))
        if sig_mixed_effects.empty:
            display(Markdown("_No mixed-effects terms reached p < .05._"))
        else:
            display_df(sig_mixed_effects, rows=80)
    else:
        mixed_effects_table = pd.DataFrame()
        display(Markdown("_Mixed-effects model could not be fit._"))
    for msg in mixed_messages:
        display(Markdown(f"_Mixed model note: {msg}_"))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), squeeze=False)
    ax0, ax1 = axes.ravel()
    sns.boxplot(data=subject_scr_neural_coupling, x="Domain", y="r_neural_scr", hue="Group", palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax0, fliersize=0)
    sns.stripplot(data=subject_scr_neural_coupling, x="Domain", y="r_neural_scr", hue="Group", dodge=True, alpha=0.55, color="black", ax=ax0, legend=False)
    ax0.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax0.set_title("Subject-level SCR-neural coupling")
    ax0.set_ylabel("Within-subject r")
    ax0.set_xlabel("")
    if ax0.get_legend() is not None:
        ax0.legend(title="Group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
    if not stats_table.empty and "sig" in stats_table.columns:
        y_min, y_max = ax0.get_ylim()
        y_text = y_max - 0.06 * (y_max - y_min)
        domain_order = [tick.get_text() for tick in ax0.get_xticklabels()]
        for _, row in stats_table.iterrows():
            sig = row.get("sig", "")
            domain = str(row.get("Domain", ""))
            if sig and domain in domain_order:
                ax0.text(domain_order.index(domain), y_text, sig, ha="center", va="top", fontsize=14, fontweight="bold", color="black")
    if not trial_scr_neural.empty:
        sns.scatterplot(data=trial_scr_neural, x="SCR_Anticipatory", y="Neural_Trajectory_Score", hue="Group", style="Domain", alpha=0.35, ax=ax1)
        for group, color in [("SAD", '#c44e52'), ("HC", '#4c72b0')]:
            gdf = trial_scr_neural[trial_scr_neural["Group"].astype(str).str.upper().eq(group)]
            if len(gdf) > 5:
                sns.regplot(data=gdf, x="SCR_Anticipatory", y="Neural_Trajectory_Score", scatter=False, ax=ax1, color=color, label=f"{group} fit")
        ax1.set_title("Descriptive trial-level SCR vs neural trajectory score")
        ax1.set_xlabel("Anticipatory SCR")
        ax1.set_ylabel("Neural trajectory score")
        handles, labels = ax1.get_legend_handles_labels()
        dedup = {}
        for handle, label in zip(handles, labels):
            if label and not str(label).startswith("_") and label not in dedup:
                dedup[label] = handle
        if dedup:
            ax1.legend(list(dedup.values()), list(dedup.keys()), title="Group / domain", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
    else:
        ax1.axis("off")
    fig.tight_layout(w_pad=4.0)
    save_current_fig(f"{PIPELINE_NAME.lower()}_trialwise_scr_neural_coupling.png")
    plt.show()
    if not trial_scr_neural.empty:
        domains = [d for d in ["Safety", "Threat", "Threat Shock Target"] if d in set(trial_scr_neural["Domain"].dropna().astype(str))]
        if domains:
            fig_corr, axes_corr = plt.subplots(1, len(domains), figsize=(6.2 * len(domains), 4.8), squeeze=False)
            for ax_corr, domain in zip(axes_corr.ravel(), domains):
                ddf = trial_scr_neural[trial_scr_neural["Domain"].astype(str).eq(domain)].copy()
                ddf["SCR_Anticipatory"] = pd.to_numeric(ddf["SCR_Anticipatory"], errors="coerce")
                ddf["Neural_Trajectory_Score"] = pd.to_numeric(ddf["Neural_Trajectory_Score"], errors="coerce")
                ddf = ddf.dropna(subset=["SCR_Anticipatory", "Neural_Trajectory_Score"])
                if len(ddf) >= 3:
                    r_all, p_all = stats.pearsonr(ddf["SCR_Anticipatory"], ddf["Neural_Trajectory_Score"])
                else:
                    r_all, p_all = np.nan, np.nan
                corr_sig = _p_to_stars(p_all)
                sns.scatterplot(data=ddf, x="SCR_Anticipatory", y="Neural_Trajectory_Score", hue="Group", palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, alpha=0.35, ax=ax_corr)
                if len(ddf) >= 3:
                    sns.regplot(data=ddf, x="SCR_Anticipatory", y="Neural_Trajectory_Score", scatter=False, ax=ax_corr, color="black", line_kws={"linewidth": 2})
                ax_corr.set_title(f"Descriptive {domain}: SCR-neural correlation {corr_sig}\nr={r_all:.3f}, p={p_all:.3g}, n={len(ddf)}")
                ax_corr.set_xlabel("Anticipatory SCR")
                ax_corr.set_ylabel("Neural trajectory score")
                if ax_corr.get_legend() is not None:
                    ax_corr.legend(title="Group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
            fig_corr.tight_layout(w_pad=4.0)
            save_current_fig(f"{PIPELINE_NAME.lower()}_trialwise_scr_neural_direct_correlation.png")
            plt.show()
    if not trial_scr_neural.empty:
        display(Markdown("**Matched trial-level rows used for coupling**"))
        display_df(trial_scr_neural, rows=40)
else:
    display(Markdown("_Trial-wise SCR-neural coupling could not be computed: missing `df_scr_trials` or trajectory data (`data_safe` / `data_threat` / `data_threat_shock`)._"))


## 6c. Placebo-Only Trial-Wise Mixed-Effects Models
This cell repeats the trial-wise mixed-effects test within the placebo arm only, including combined safety/threat and Safety-only / Threat-only follow-up models.


In [ ]:
display(Markdown("**Placebo-only trial-wise mixed-effects model**"))
display(Markdown("Placebo-only model: z(SCR) ~ z(neural trajectory score) * Group * Domain + trial + subject random effects. Safety-only and Threat-only follow-ups use z(SCR) ~ z(neural trajectory score) * Group + trial + subject random effects."))

placebo_mixed_tables = []
placebo_mixed_messages = []
if "trial_scr_neural" not in globals() or not isinstance(trial_scr_neural, pd.DataFrame) or trial_scr_neural.empty:
    display(Markdown("_Placebo-only mixed model could not be fit because `trial_scr_neural` is unavailable._"))
elif "_fit_trialwise_mixedlm" not in globals() or not callable(_fit_trialwise_mixedlm):
    display(Markdown("_Placebo-only mixed model could not be fit because the mixed-model helper was not defined. Run section 6b first._"))
elif "Drug" not in trial_scr_neural.columns:
    display(Markdown("_Placebo-only mixed model could not be fit because the trial-wise table has no `Drug` column._"))
else:
    placebo_df = trial_scr_neural[trial_scr_neural["Drug"].astype(str).str.lower().eq("placebo")].copy()
    display(Markdown(f"Placebo trial rows: **{len(placebo_df)}**; subjects: **{placebo_df['sub_ID'].nunique() if 'sub_ID' in placebo_df.columns else 'NA'}**"))
    if placebo_df.empty:
        display(Markdown("_No placebo-only rows were available for the placebo-focused mixed model._"))
    else:
        placebo_formula = "SCR_Z ~ Neural_Z * C(Group) * C(Domain) + Trial_Z"
        table, msg = _fit_trialwise_mixedlm(placebo_df, "Placebo only: combined safety/threat", placebo_formula)
        if not table.empty:
            placebo_mixed_tables.append(table)
        if msg:
            placebo_mixed_messages.append(msg)
        for domain, domain_df in placebo_df.groupby("Domain"):
            domain_formula = "SCR_Z ~ Neural_Z * C(Group) + Trial_Z"
            table, msg = _fit_trialwise_mixedlm(domain_df, f"Placebo only: {domain} only", domain_formula)
            if not table.empty:
                placebo_mixed_tables.append(table)
            if msg:
                placebo_mixed_messages.append(msg)
        if placebo_mixed_tables:
            placebo_mixed_effects_table = pd.concat(placebo_mixed_tables, ignore_index=True)
            display_df(placebo_mixed_effects_table, rows=120)
            placebo_sig_mixed_effects = placebo_mixed_effects_table[pd.to_numeric(placebo_mixed_effects_table["p"], errors="coerce") < 0.05].copy()
            display(Markdown("**Significant placebo-only mixed-effects terms (p < .05)**"))
            if placebo_sig_mixed_effects.empty:
                display(Markdown("_No placebo-only mixed-effects terms reached p < .05._"))
            else:
                display_df(placebo_sig_mixed_effects, rows=80)
        else:
            placebo_mixed_effects_table = pd.DataFrame()
            display(Markdown("_Placebo-only mixed-effects model could not be fit._"))
        for msg in placebo_mixed_messages:
            display(Markdown(f"_Placebo mixed model note: {msg}_"))


## 7. Decision Boundary Characteristics
Decision-boundary and self-network uncertainty statistics are listed and plotted from saved results where available.


In [ ]:
r15 = results.get("stage15_decision", {})
show_figures_by_keywords("Decision boundary characteristics", "analysis_14", "results_14", "decision", "uncertainty", "boundary", max_figs=12)
display_df(flatten_result_dict(r15), rows=80)
for key in ["df_stats", "df_decision", "decision_df", "summary_df", "df_boundary"]:
    df = r15.get(key) if isinstance(r15, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Decision boundary: {key}")


## 8. Safety Restoration and Threat Discrimination
This section summarizes later-stage safety restoration and threat discrimination outputs.


In [ ]:
r16 = results.get("stage16_safety_threat", {})
show_figures_by_keywords("Safety restoration and threat discrimination", "analysis_21", "results_21", "safety", "threat", "restoration", "discrimination", max_figs=12)
display_df(flatten_result_dict(r16), rows=80)
for key in ["df_stats", "df_plot", "df_safety_threat", "summary_df"]:
    df = r16.get(key) if isinstance(r16, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Safety/Threat: {key}")


## 9. Drift Efficiency for Safety and Threat Maintenance
This section reports drift-efficiency statistics and plots saved or reconstructed from the stage outputs.


In [ ]:
r18 = results.get("stage18_drift_efficiency", {})
show_figures_by_keywords("Drift efficiency", "analysis_22", "results_22", "drift_efficiency", "maintenance", max_figs=12)
display_df(flatten_result_dict(r18), rows=80)
for key in ["df_stats", "df_plot", "df_efficiency", "summary_df"]:
    df = r18.get(key) if isinstance(r18, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Drift efficiency: {key}")


## 9b. Second Threat Metric: Shock-Target Drift Efficiency

In [ ]:
# Second threat metric visualization: drift efficiency toward shock/US target pattern.
def _find_drift_payload(obj, max_depth=8):
    if max_depth < 0:
        return None
    if isinstance(obj, dict):
        df = obj.get("df")
        if isinstance(df, pd.DataFrame) and "Domain" in df.columns and df["Domain"].astype(str).str.contains("Shock", case=False, na=False).any():
            return obj
        for val in obj.values():
            found = _find_drift_payload(val, max_depth - 1)
            if found is not None:
                return found
    elif isinstance(obj, (list, tuple)):
        for val in obj:
            found = _find_drift_payload(val, max_depth - 1)
            if found is not None:
                return found
    return None

drift_payload = _find_drift_payload(results) if isinstance(results, dict) else None
if drift_payload is None:
    display(Markdown("_No shock-target drift-efficiency rows found yet. Re-run drift efficiency after the shock-target update._"))
else:
    df_drift_shock_source = drift_payload.get("df")
    drift_stats = drift_payload.get("stats", {}) if isinstance(drift_payload, dict) else {}
    shock_labels = drift_payload.get("shock_target_labels", []) if isinstance(drift_payload, dict) else []
    shock_rows = df_drift_shock_source[df_drift_shock_source["Domain"].astype(str).str.contains("Shock", case=False, na=False)].copy()
    display(Markdown(f"**Shock-target labels used by the `.py` script:** `{shock_labels}`"))
    display(Markdown("**Threat shock-target drift-efficiency rows**"))
    display_df(shock_rows, rows=50)
    shock_stat_rows = []
    for key, val in drift_stats.items():
        if "Shock" in str(key):
            shock_stat_rows.append({"Domain_Metric": key, "interaction_p": val})
    shock_stats_table = pd.DataFrame(shock_stat_rows)
    display(Markdown("**Threat shock-target drift-efficiency statistics**"))
    display_df(shock_stats_table, rows=20)
    if not shock_rows.empty:
        sns.set_context("poster", font_scale=0.85)
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), squeeze=False)
        pal_group = {"SAD": "#c44e52", "HC": "#4c72b0"}
        for ax, metric in zip(axes.ravel(), ["Cosine", "Projection"]):
            mcol = metric if metric in shock_rows.columns else metric.lower() if metric.lower() in shock_rows.columns else None
            if mcol is None:
                ax.axis("off")
                ax.set_title(f"Missing {metric}")
                continue
            p_val = drift_stats.get(f"Threat Shock Target_{metric}", np.nan)
            sns.pointplot(data=shock_rows, x="Drug", y=mcol, hue="Group", palette=pal_group, order=["Placebo", "Oxytocin"], hue_order=["SAD", "HC"], dodge=0.2, markers=["o", "s"], capsize=0.1, errorbar="se", ax=ax)
            sns.stripplot(data=shock_rows, x="Drug", y=mcol, hue="Group", palette=pal_group, order=["Placebo", "Oxytocin"], hue_order=["SAD", "HC"], dodge=True, alpha=0.35, jitter=True, legend=False, ax=ax)
            ax.axhline(0, color="gray", linestyle="--", linewidth=1)
            ax.set_title(f"Threat Shock Target: {metric}\nGroup x Drug p={p_val:.3g}" if np.isfinite(p_val) else f"Threat Shock Target: {metric}")
            handles, labels = ax.get_legend_handles_labels()
            dedup = {}
            for handle, label in zip(handles, labels):
                if label not in dedup:
                    dedup[label] = handle
            ax.legend(list(dedup.values()), list(dedup.keys()), title="Group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
        fig.tight_layout(w_pad=4.0)
        save_current_fig(f"{PIPELINE_NAME.lower()}_threat_shock_target_drift_efficiency.png")
        plt.show()


## 10. Probabilistic Opening / Decision-Probability Extraction
This section summarizes probabilistic opening and decision-probability extraction outputs.


In [ ]:
r19 = results.get("stage19_prob_opening", {})
show_figures_by_keywords("Probabilistic opening", "analysis_23", "results_23", "prob", "opening", "decision_probability", max_figs=12)
display_df(flatten_result_dict(r19), rows=80)
for key in ["df_stats", "df_plot", "df_prob", "summary_df"]:
    df = r19.get(key) if isinstance(r19, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Probabilistic opening: {key}")


## 11. Spatial Re-Alignment
Spatial re-alignment statistics and figures are displayed here.


In [ ]:
r20 = results.get("stage20_realignment", {})
show_figures_by_keywords("Spatial re-alignment", "analysis_24", "results_24", "realignment", "re-alignment", "alignment", max_figs=12)
display_df(flatten_result_dict(r20), rows=80)
for key in ["df_stats", "df_plot", "df_realignment", "summary_df"]:
    df = r20.get(key) if isinstance(r20, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Spatial realignment: {key}")


## 12. Reverse Cross-Decoding
Reverse cross-decoding statistics and figures are displayed here.


In [ ]:
r21 = results.get("stage21_reverse", {})
show_figures_by_keywords("Reverse cross-decoding", "analysis_25", "results_25", "reverse", "cross_decoding", "cross-decoding", max_figs=12)
display_df(flatten_result_dict(r21), rows=80)
for key in ["df_stats", "df_plot", "df_reverse", "summary_df"]:
    df = r21.get(key) if isinstance(r21, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        maybe_plot_numeric_bars(df, f"Reverse cross-decoding: {key}")


## 13. Group-Wise Neural-Clinical Pearson Correlations
Pearson correlation tables and heatmaps are shown for group-wise neural-clinical associations.


In [ ]:
r27 = results.get("stage27_pearson", {})
show_figures_by_keywords("Group-wise Pearson correlations", "analysis_27", "results_27", "pearson", "correlation", "clinical", max_figs=12)
display_df(flatten_result_dict(r27), rows=80)
df = extract_first_dataframe(r27, preferred=("df_res_grp", "df_pearson", "df_neural_clinical_pearson", "results_df"))
plot_corr_table(df, "Group-wise neural-clinical Pearson correlations")


## 14. Clinical Scores, Neural Indices, and Master Merge
The request listed section 14 without a label, so this notebook uses it as the audit point for clinical-score loading, neural-index extraction, and the merged analysis table used by the correlation/regression stages.


In [ ]:
for name in ["stage23_clinical_scores", "stage24_neural_indices", "stage26_master_merge"]:
    display(Markdown(f"**{name}**"))
    res = results.get(name, {})
    display_df(flatten_result_dict(res), rows=50)
    df = extract_first_dataframe(res, preferred=("df_master", "df_master_analysis", "df_scores", "df_indices", "df_neural"))
    if isinstance(df, pd.DataFrame):
        display_df(df, rows=20)
        maybe_plot_numeric_bars(df, name.replace("_", " "))


## 15. Group-Wise Partial Correlations Adjusted for Covariates
Partial correlation tables and heatmaps are shown when the saved stage contains adjusted association estimates.


In [ ]:
r28 = results.get("stage28_partial", {})
show_figures_by_keywords("Group-wise partial correlations", "analysis_28", "results_28", "partial", "covariate", "clinical", max_figs=12)
display_df(flatten_result_dict(r28), rows=80)
df = extract_first_dataframe(r28, preferred=("df_res_partial", "df_partial", "df_res_grp", "results_df"))
plot_corr_table(df, "Group-wise partial correlations adjusted for covariates")


## 16. Outlier Removal and Z-Scoring
This section audits outlier removal and z-scoring for neural, clinical, and covariate variables.


In [ ]:
r29 = results.get("stage29_zscore_outlier", {})
show_figures_by_keywords("Outlier removal and z-scoring", "analysis_29", "results_29", "outlier", "zscore", "z_score", max_figs=12)
display_df(flatten_result_dict(r29), rows=80)
for key in ["df_outlier_summary", "outlier_summary", "df_master_analysis_z", "df_z", "df_zscored"]:
    df = r29.get(key) if isinstance(r29, dict) else None
    if isinstance(df, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(df, rows=40)
        numeric = df.select_dtypes(include=[np.number])
        if not numeric.empty and numeric.shape[1] <= 30:
            fig, ax = plt.subplots(figsize=(max(7, 0.4 * numeric.shape[1]), 4))
            sns.boxplot(data=numeric, ax=ax, color="#9C755F")
            ax.set_title(key)
            ax.tick_params(axis="x", rotation=45)
            save_current_fig(f"section16_{key}.png")
            plt.show()


## 17. Z-Scored OLS / Regression Plots for Neural-Clinical Associations
This final section lists z-scored OLS models and plots regression coefficients or saved regression figures where available.


In [ ]:
r30 = results.get("stage30_ols", {})
show_figures_by_keywords("Z-scored OLS regression", "analysis_30", "results_30", "ols", "regression", "zscored", "z_score", max_figs=12)
display_df(flatten_result_dict(r30), rows=80)
df = extract_first_dataframe(r30, preferred=("df_ols_results", "df_ols", "results_df", "df_regression"))
if isinstance(df, pd.DataFrame):
    display_df(df, rows=60)
    beta_col = next((c for c in df.columns if c.lower() in {"beta", "coef", "estimate", "standardized_beta"}), None)
    p_col = next((c for c in df.columns if c.lower() in {"p", "pval", "p_value", "p_uncorrected"}), None)
    term_col = next((c for c in df.columns if c.lower() in {"term", "predictor", "neural_metric", "index", "model"}), None)
    if beta_col:
        plot_df = df.copy()
        if term_col is None:
            plot_df["term"] = np.arange(len(plot_df))
            term_col = "term"
        plot_df["term_label"] = plot_df[term_col].astype(str)
        fig, ax = plt.subplots(figsize=(max(8, 0.35 * len(plot_df)), 4.5))
        sns.barplot(data=plot_df, x="term_label", y=beta_col, ax=ax, color="#E15759")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title("Z-scored OLS coefficients")
        ax.tick_params(axis="x", rotation=60)
        save_current_fig("section17_z_ols_coefficients.png")
        plt.show()
    if beta_col and p_col:
        plot_df = df.copy()
        plot_df["neg_log10_p"] = -np.log10(pd.to_numeric(plot_df[p_col], errors="coerce"))
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(data=plot_df, x=beta_col, y="neg_log10_p", ax=ax)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title("Effect size versus significance")
        save_current_fig("section17_z_ols_beta_vs_p.png")
        plt.show()
else:
    display(Markdown("_No OLS dataframe found in saved payload._"))
